# Deep Learning Statistical Arbitrage – Implémentation de Fourier + FFN

## 1. Référence de l'article

**Titre** : *Deep Learning Statistical Arbitrage*  
**Auteurs** : Jorge Guijarro-Ordonez, Markus Pelger, Greg Zanotti  
**Institution** : Stanford University  
**Liens** :  
- [SSRN](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3862004)  
- [arXiv](https://arxiv.org/abs/2106.04028)  
- [GitHub officiel](https://github.com/gregzanotti/dlsa-public)

---

## 2. Contexte et motivations

L’arbitrage statistique est une **méthode de trading** qui repose sur l'hypothèse que des actifs jugés similaires ont des prix qui évoluent de manière proche en moyenne. Concrètement, on identifie deux actifs considérés comme substituables (par exemple General Motors et Ford), on construit un portefeuille long-short en achetant l'actif sous-évalué et en vendant l'actif surévalué, et on attend un retour à l'équilibre pour déboucler les positions avec un profit. La stratégie parie sur le fait que les écarts de prix sont temporaires et finissent par se résorber.

Cette approche repose donc sur trois piliers :

1. **Construire un portefeuille long-short**  
   On définit l’écart de rendement entre deux actifs, par exemple :  
   $\epsilon_t = R_t^{GM} - R_t^{Ford}$  
   Une position longue sur l'actif dont le rendement est anormalement bas et une position short sur l'actif dont le rendement est anormalement haut.

2. **Extraire un signal de trading**  
   On modélise $\epsilon_t$ pour détecter les moments où l’écart s'écarte significativement de sa moyenne historique, signalant une opportunité d'arbitrage.

3. **Décider d’une allocation**  
   On détermine la taille des positions à prendre (degrés d'exposition longs et shorts), dans le but de maximiser le gain final sous contrainte de risque :  
   $\max \mathbb{E}[\text{payoff}_T]$


Toutefois, dans la pratique, on se heurte à 3 difficultés majeures:

**Premier défi : le grand nombre d’actifs aux similarités inconnues.**  
Comment identifier automatiquement quelles actions sont suffisamment proches pour être arbitrées ? Dans le cadre de cet article, les modèles factoriels sont présentés à cet effet : deux actifs ayant des expositions similaires à des facteurs de risque communs sont considérés comme “similaires”.

**Deuxième défi : la complexité des motifs temporels.**  
Le retour à la moyenne des prix n’est pas toujours aussi simple qu’un processus linéaire. Il peut y avoir des tendances temporaires plus complexes, des asymétries entre les phases de hausse et de baisse. Des phénomènes que les modèles paramétriques classiques que nous connaissons notamment celui de OU peinent à capturer.

**Troisième défi : l’allocation optimale dépend de l’objectif de trading.**  
Dans le contexte de l'arbitrage statistique, il n’est pas seulement question de prédire le signe du prochain mouvement. Il faut pouvoir tenir compte du compromis risque/rendement, d’éventuelles contraintes de levier, des coûts de transaction qu'entraineraient des prises de position. La “meilleure” règle n’est donc pas universelle.

Face à ces difficultés, les méthodes d’apprentissage profond ont été priviligiés par les auteurs, principalement parcequ'elles sont flexibles, capables de traiter de grandes quantités de données et de détecter des motifs complexes.

<u>Mais tout l’enjeu ici est de **poser correctement le problème d’estimation** : il ne s’agit pas d’un problème de prédiction classique (prédire le résidu) mais d’un problème de trading. C’est pourquoi l’article propose d’utiliser directement une **fonction objectif liée au trading** (le Sharpe ratio) appliquée aux résidus d’un modèle factoriel.</u>

Trois questions centrales guident alors l’analyse :

1. Quelle est la “meilleure solution” pour les trois éléments clés (portefeuilles, signal, allocation) ?
2. Qu’est-ce qui compte vraiment pour rendre l’arbitrage rentable ?
3. Combien d’arbitrage réaliste existe-t-il réellement sur les marchés ?

---





## 3. Cadre de modélisation : formalisation des 3 pilliers de l'arbitage statistique

Comme nous l'avons stipulé précédemment, l’arbitrage statistique repose sur trois piliers : la construction d’un portefeuille long-short, l’extraction d’un signal de trading, et la décision d’allocation. Dans cette section,il s’agit maintenant de les formaliser mathématiquement.

---

### 3.1 Premier pilier : la construction des portefeuilles d’arbitrage

L’objectif est de construire des portefeuilles long-short à partir d’un grand nombre d’actifs, sans avoir à définir manuellement des paires. Pour cela, on utilise un modèle factoriel.

**Modèle factoriel conditionnel**

On note $R_{n,t}$ le rendement excédentaire (rendement moins taux sans risque) de l’action $n$ à la date $t$. Le nombre d’actifs $N_t$ peut varier dans le temps. L’article suppose que les rendements suivent un modèle factoriel conditionnel :

$$R_{n,t} = \beta_{n,t-1}^{\top} F_t + \epsilon_{n,t}$$

- $F_t$ est un vecteur de $K$ facteurs qui capturent le risque systématique.
- $\beta_{n,t-1}$ est un vecteur (expositions au risque) qui dépend de l’information disponible à la date $t-1$. Ces coefficients peuvent donc varier dans le temps.
- $\epsilon_{n,t}$ est le résidu, c’est-à-dire la part du rendement non expliquée par les facteurs.

**La notion de similarité**

Deux actifs sont considérés comme similaires s’ils ont la même exposition aux facteurs de risque ($\beta$ identiques). Ils devraient alors avoir la même valeur fondamentale. Le résidu $\epsilon_{n,t}$ représente l’écart temporaire par rapport à cette valeur fondamentale. **C’est donc sur elle que se fonde l’arbitrage**.

**Les modèles factoriels retenus**

L’article retient les trois familles de modèles factoriels les plus importantes empiriquement d'après les auteurs :

1. **Facteurs observés** : les facteurs de Fama-French
2. **Facteurs statistiques** : une analyse en composantes principales (PCA) sur la matrice des rendements.
3. **Facteurs conditionnels** : les facteurs IPCA (Instrumented PCA) où les coefficients d'exposition au risque $\beta_{n,t-1}$ dépendent des caractéristiques des entreprises.

**Des facteurs aux portefeuilles d’arbitrage**

Sans perte de généralité, les auteurs supposent que l'on peut traiter les facteurs comme des rendements d’actifs tradables. Et, même au cas où ils ne le sont pas, on construit des portefeuilles miroir  en les projetant sur l’espace des actifs :

$$F_t = w_{t-1}^{F \top} R_t$$

En remplaçant $F_t$ par cette expression dans la définition du résidu, on obtient :

$$\epsilon_t = R_t - \beta_{t-1}^{\top} F_t = R_t - \beta_{t-1}^{\top} w_{t-1}^{F} R_t = \underbrace{\left( I_{N_t} - \beta_{t-1}^{\top} w_{t-1}^{F} \right)}_{\Phi_{t-1}} R_t$$

La matrice $\Phi_{t-1}$ est entièrement déterminée par les facteurs et les $\beta$. Le vecteur $\epsilon_t$ alors représente les rendements d’un ensemble de **portefeuilles tradables**, que l’on appelle portefeuilles d’arbitrage.

**Propriétés des portefeuilles d’arbitrage**

Ces portefeuilles présentent quatre propriétés importantes :

- Ils sont **tradables** : ce sont des combinaisons linéaires des actifs de base.
- Ils sont **neutres vis-à-vis des facteurs** : on voit bien que toute exposition au risque systématique a été retirée.
- Ils sont **faiblement corrélés entre eux** : une fois les facteurs retirés, il ne reste plus que des dépendances résiduelles faibles.
- La théorie de l’arbitrage (APT) implique que l’espérance des résidus est nulle : $\mathbb{E}[\epsilon_{n,t}] = 0$ (lorsque le modèle est bien spécifié),(les auteurs ont pu le confirmer empiriquement) donc tout écart doit finir par se résorber. Tout écart par rapport à zéro est donc temporaire par nature, ce qui justifie le pari sur le retour à la moyenne.





<font color="red">

P11 remarque: Les auteurs reconnaissent que leur modèle factoriel peut être imparfait (certains facteurs de risque ne sont pas pris en compte). Mais cela n'invalide pas leur approche. Pourquoi ? Parce que le portefeuille d'arbitrage n'est jamais un "pur" pari sur un retour à zéro. C'est un pari sur l'écart entre l'actif cible et son **portefeuille miroir** (mimicking portfolio). Si le modèle omet certains facteurs, le portefeuille miroir ne capte qu'une partie du risque systématique. Les résidus reflètent alors les déviations par rapport à **ces facteurs capturés seulement**. On trade donc ces déviations, sans garantir un retour à la moyenne absolu. L'article ne promet pas un arbitrage sans risque ; il promet une stratégie qui, empiriquement, génère des rendements ajustés du risque élevés malgré ces imperfections.

</font>

---

### 3.2. Deuxième et troisième piliers : signal d’arbitrage et allocation

Une fois les résidus obtenus, l’arbitrage statistique se décompose en deux étapes. On note $\epsilon_{t-L+1}^{t}$ la fenêtre glissante des $L$ derniers résidus (avec $L = 30$ jours).

**Notation préalable : la filtration**

On désigne par $\mathcal{F}_{t-1}$ l’ensemble de toute l’information disponible à la date $t-1$ : les rendements passés $R_{t-1}$ (incluant les facteurs) ainsi que les informations qui déterminent les coefficients $\beta_{t-1}$.

**Première étape : la fonction de signal**

La fonction de signal $\theta \in \Theta$ modélise la structure temporelle des résidus à partir des $L$ dernières observations et produit une statistique suffisante pour la décision de trading :

$$\theta : \epsilon_{t-L+1}^{t} \longrightarrow \theta_{t-1}$$

Le signal $\theta_{t-1}$ est une **statistique suffisante** pour la politique de trading : toute l’information pertinente pour la décision y est résumée.

Cette formulation repose sur deux hypothèses implicites :

1. **Stationnarité conditionnelle** : la série temporelle des résidus suit une distribution stationnaire conditionnellement à ses valeurs passées. Ce qui permet une modélisation dans un cadre assez général qui inclut les principaux modèles de séries financières.
2. **Statistique suffisante** : les $L$ derniers résidus contiennent toute l’information nécessaire pour obtenir le signal $\theta_{t-1}$. Cela reflète ainsi l’idée que l’arbitrage est une déviation temporaire du prix "fair".

**Deuxième étape : la fonction d’allocation**

La fonction d’allocation $w \in \mathcal{W}$ transforme ce signal en un poids d’investissement :

$$w : \theta_{t-1} \longrightarrow w_{t-1}^{\epsilon}$$

La sortie $w_{t-1}^{\epsilon}$ est un scalaire : une valeur positive signifie une position longue (on s’attend à une hausse du résidu), une valeur négative une position courte (on s’attend à une baisse).

---

#### L'estimation

L’estimation s’effectue en maximisant le rendement ajusté du risque, pour une classe de modèles donnée. Étant donné une fonction d’utilité concave $U(\cdot)$, le problème s’écrit :

$$\max_{w \in \mathcal{W}, \theta \in \Theta} \mathbb{E}_{t-1} \left[ U\left( w_{t-1}^{R \top} R_t \right) \right]$$

sous les contraintes :

$$w_{t-1}^{R} = \frac{w_{t-1}^{\epsilon \top} \Phi_{t-1}}{\| w_{t-1}^{\epsilon \top} \Phi_{t-1} \|_1} \quad \text{et} \quad w_{t-1}^{\epsilon} = w^{\epsilon}\big(\theta(\epsilon_{t-L+1}^{t})\big)$$

En présence de coûts de transaction, on calcule l’utilité espérée du portefeuille **net** des coûts de transaction.

L’article se concentre sur deux objectifs principaux :

- **Maximisation du Sharpe ratio** :

$$\max_{w \in \mathcal{W}, \theta \in \Theta} \frac{\mathbb{E}[w_{t-1}^{R \top} R_t]}{\sqrt{\text{Var}(w_{t-1}^{R \top} R_t)}}$$

- **Objectif moyenne-variance** (pour un paramètre d’aversion au risque $\gamma$) :

$$\max_{w \in \mathcal{W}, \theta \in \Theta} \mathbb{E}[w_{t-1}^{R \top} R_t] - \gamma \cdot \text{Var}(w_{t-1}^{R \top} R_t)$$

La somme des poids $w_{t-1}^{R}$ en valeur absolue est égale à 1 : c'est une contrainte de levier implicite (on empêche que des positions trop grandes puissent êtres prises). Signal et allocation sont optimisés conjointement, mais la décomposition n'est pas unique (plusieurs couples $\theta$ et $w$ peuvent donner la même stratégie). Toutefois, elle peut également se faire séparément, la plupart des modèles classiques estiment d'abord le signal (paramètres d'un modèle, moments d'une distribution, filtre temporel), puis l'allocation dans un second temps.

---


---

## 4. Les trois classes de modèles retenues

**A ce niveau, on admet que les résidus sont déjà donnés**

Dans cette section, on considère les résidus comme déjà calculés, on a déjà choisi un modèle factoriel et on a obtenu les séries de résidus $\epsilon_{n,t}$.

Les auteurs font comme si l'on tradait les résidus directement; comme s'ils étaient des actifs. Etant donné que, en pratique, on ne trade pas les résidus directement : on les reconvertit en positions sur les actions originales via la matrice $\Phi_{t-1}$ (vue dans la section 3.1).

En outre, l'entrée des fonctions d'extraction du signal est constituée des **$L$ derniers résidus cumulés**. On note :
<font color="red">
$$x := \text{Int}\left(\epsilon_{n,t-1}^{L}\right) = \left( \epsilon_{n,t-L},\ \epsilon_{n,t-L-1} + \epsilon_{n,t-L},\ \dots,\ \sum_{l=1}^{L} \epsilon_{n,t-L-1+l} \right)$$
</font>

L'article compare trois grandes familles de modèles, qui se distinguent par la manière dont elles implémentent le signal $\theta$ et l'allocation $w^{\epsilon}$.

| Classe | Signal $\theta$ | Allocation $w^{\epsilon}$ |
|--------|----------------|--------------------------|
| **1. Paramétrique** | Modèle de réversion (Ornstein-Uhlenbeck) + $R^2$ | Règle de seuillage |
| **2. Filtre pré-spécifié + réseau** | Filtre fréquentiel (FFT) | FFN non-paramétrique |
| **3. Deep learning complet** | CNN + Transformer (appris) | FFN (apprise) |

---


---
#### 4.1 Première classe : modèles paramétriques (OU + Threshold)

Dans cette approche, chaque résidu cumulé $x$ est modélisé comme un processus d'Ornstein-Uhlenbeck (OU) :

$$dX_t = \kappa(\mu - X_t)dt + \sigma dB_t$$

Les paramètres $\hat{\kappa}$, $\hat{\mu}$, $\hat{\sigma}$ sont estimés à partir des moments de la série discrétisée. Le signal intègre également le dernier résidu cumulé $X_L$ et la mesure de qualité d'ajustement $R^2$ (coefficient de détermination):

$$\theta_{OU} = (\hat{\kappa}, \hat{\mu}, \hat{\sigma}, X_L, R^2)$$

L'allocation ici définie par la règle de seuillage suivante :

$$w^{\epsilon}(\theta_{OU}) =
\begin{cases}
-1 & \text{si } \frac{X_L - \hat{\mu}}{\hat{\sigma}/\sqrt{2\hat{\kappa}}} > c_{thresh} \text{ et } R^2 > c_{crit} \\
1 & \text{si } \frac{X_L - \hat{\mu}}{\hat{\sigma}/\sqrt{2\hat{\kappa}}} < -c_{thresh} \text{ et } R^2 > c_{crit} \\
0 & \text{sinon}
\end{cases}$$

Les paramètres de seuillage $c_{thresh}$ et $c_{crit}$ sont des hyperparamètres à determiner.
La stratégie fonctionne ainsi : on achète ou on vend le résidu en fonction du ratio $\frac{X_L - \mu}{\sigma/\sqrt{2\kappa}}$. Si ce ratio dépasse un seuil, il est probable que le processus revienne vers sa moyenne de long terme, ce qui déclenche le trade. En revanche, si le $R^2$ est trop faible, les prédictions du modèle sont jugées peu fiables, et on s'abstient de trader.

Toutefois, le modèle  peut être mal spécifié (présence de tendances, fréquences de réversion multiples, etc.). Une autre limitation reside aussi en le fait que sa fonction d'allocation est  trop rigide.


---


#### 4.2 Deuxième classe : filtre pré-spécifié avec réseau de neurones (FFT + FFN)

Un filtre linéaire pré-spécifié ("time-invariant") s'écrit :

$$\theta_l = \sum_{j=1}^{L} W^{filter}_{j} x_j$$

Cette formulation inclut les alors modèles tels que le ARMA etc. C'est donc une transformation d'une série temporelle qui fournit une représentation alternative mettant en évidence certains motifs dynamiques.

Parmi ces filtres, les auteurs spécifient que les **filtres fréquentiels** sont les plus adaptés aux motifs de retour à la moyenne. Ils sont donc les mieux adaptés à notre contexte.
A cet effet, la transformée de Fourier (FFT) est utilisée; elle décompose la série en une somme de processus de réversion de différentes fréquences :

$$x_l = a_0 + \sum_{j=1}^{L/2-1} \left( a_j \cos\left(\frac{2\pi j}{L} l\right) + b_j \sin\left(\frac{2\pi j}{L} l\right) \right) + a_{L/2} \cos(\pi l)$$

Le signal est le vecteur des coefficients, interprétés comme des expositions à des motifs de réversion long et court terme :

$$\theta_{FFT}(x) = (a_0, \dots, a_{L/2}, b_1, \dots, b_{L/2-1})$$

Vu que le FFT est une transformation **inversible**, elle ne perd donc aucune information. Elle représente juste la série sous une forme où les relations fréquentielles deviennent explicites.

L'allocation est assurée par un FFN non paramétrique :

$$w^{\epsilon}_{|FFT}(\theta_{FFT}) = g_{FFN}(\theta_{FFT})$$

Le FFN apprend essentiellement à ignorer les fréquences dont les coefficients sont faibles (considérées comme du bruit).

La FFT améliore le modèle OU car elle peut traiter des motifs de réversion multiples combinés à différentes fréquences. Toutefois, le choix d'un filtre pré-spécifié limite les motifs exploitables : la FFT échoue si les données suivent un motif qui ne peut pas être bien approché par un petit nombre de cosinus et sinus. Pour illustrer l'importance du filtre fréquentiel, les auteurs testent également le filtre identité $\theta_{ident}(x) = x$ (résidus bruts). Ils trouvent des performances moins bonnes : un FFN seul a du mal à apprendre des dépendances temporelles complexes, alors que la représentation fréquentielle rend l'apprentissage plus efficace.

---

## 5. Implémentation

<font color ="red">https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_5_factors_2x3.html: description des  facteurs fama franch à mettre en annexe du rapport<font>

#### Import des données

---

In [ ]:
# Installation et imports
#!pip install yfinance pandas numpy pandas-datareader -q

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

---
##### Import des facteurs Fama-franch (5)

In [ ]:
from google.colab import files
uploaded = files.upload()

# Charger le fichier nettoyé
df_factors = pd.read_csv('F-F_Research_Data_5_Factors_2x3_daily.csv')
df_factors.head()


Saving F-F_Research_Data_5_Factors_2x3_daily.csv to F-F_Research_Data_5_Factors_2x3_daily.csv


,Unnamed: 0,Mkt-RF,SMB,HML,RMW,CMA,RF
0,19630701,-0.67,0.00,-0.34,-0.01,0.16,0.01
1,19630702,0.79,-0.26,0.26,-0.07,-0.20,0.01
2,19630703,0.63,-0.17,-0.09,0.18,-0.34,0.01
3,19630705,0.40,0.08,-0.27,0.09,-0.34,0.01
4,19630708,-0.63,0.04,-0.18,-0.29,0.14,0.01


In [ ]:
# Renommer la première colonne
df_factors = df_factors.rename(columns={'Unnamed: 0': 'Date'})

# Convertir la colonne Date en datetime
df_factors['Date'] = pd.to_datetime(df_factors['Date'], format='%Y%m%d')

# Diviser les facteurs par 100 (passer de pourcentage à décimal)
factor_cols = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']
df_factors[factor_cols] = df_factors[factor_cols] / 100

# Mettre Date en index
df_factors = df_factors.set_index('Date')

In [ ]:

print(df_factors.head())
print(f"\nShape: {df_factors.shape}")
print(f"Période: {df_factors.index.min()} à {df_factors.index.max()}")

            Mkt-RF     SMB     HML     RMW     CMA      RF
Date                                                      
1963-07-01 -0.0067  0.0000 -0.0034 -0.0001  0.0016  0.0001
1963-07-02  0.0079 -0.0026  0.0026 -0.0007 -0.0020  0.0001
1963-07-03  0.0063 -0.0017 -0.0009  0.0018 -0.0034  0.0001
1963-07-05  0.0040  0.0008 -0.0027  0.0009 -0.0034  0.0001
1963-07-08 -0.0063  0.0004 -0.0018 -0.0029  0.0014  0.0001

Shape: (15770, 6)
Période: 1963-07-01 00:00:00 à 2026-02-27 00:00:00


In [ ]:
# Filtrer sur ta période 1996-2016 qui correspond à la période d'étude
df_factors_1996_2016 = df_factors['1996':'2016']
print(f"\nShape après filtrage: {df_factors_1996_2016.shape}")


Shape après filtrage: (5288, 6)


---

##### Import des données du SP 500

Dû à l'incomplétude des données SP500, notamment la forte présence de valeurs manquantes pour la période initiale considérée, nous avons été obligés par les préprocess faits en amont de nous restreindre à la période  2007_03_23 au 2016_12_30

In [ ]:
from google.colab import files
uploaded = files.upload()

df_returns = pd.read_csv('returns_sp500_2007_03_23_2016_12_30clean.csv')
df_returns = df_returns.set_index("Date")
df_returns.head()



Saving returns_sp500_2007_03_23_2016_12_30clean.csv to returns_sp500_2007_03_23_2016_12_30clean.csv


,TEL,FAST,SWK,IRM,LUV,CMI,NXPI,PODD,JCI,FRT,...,XYZ,CME,KDP,CCL,COR,WSM,MU,AVB,ABT,BK
Date,,,,,,,,,,,,,,,,,,,,,
2007-03-23,-0.001287,0.012008,0.005933,0.003764,0.001335,0.014729,0.0,-0.031955,0.003755,0.000877,...,-0.016832,-0.006047,-0.003922,0.005853,0.011448,-0.003739,-0.002582,0.001558,-0.011743,0.002210
2007-03-26,-0.001287,-0.019316,-0.009830,-0.004125,-0.008000,-0.009654,0.0,-0.031955,-0.004987,-0.019838,...,-0.016832,-0.004881,-0.003922,-0.001247,-0.001299,0.007217,0.001726,-0.020448,0.062755,0.003431
2007-03-27,-0.001287,-0.017164,0.000000,-0.012424,-0.001344,-0.019296,0.0,-0.031955,-0.019737,-0.011294,...,-0.016832,-0.001468,-0.003922,-0.014149,-0.009662,-0.002293,0.034453,-0.023673,-0.011181,-0.008548
2007-03-28,-0.001287,0.000859,-0.004152,-0.006100,-0.011440,-0.016727,0.0,-0.031955,-0.004794,-0.004184,...,-0.016832,-0.015184,-0.003922,-0.020051,0.003003,0.002298,0.000000,-0.020063,-0.014841,-0.008374
2007-03-29,-0.001287,-0.006007,0.002719,0.000000,-0.004084,0.010388,0.0,-0.031955,0.003222,-0.002499,...,-0.016832,0.003514,-0.003922,0.009907,0.000187,-0.007452,-0.014155,0.001352,0.000896,0.007451


##### Calcul des des rendements excédentaires


In [ ]:
df_returns.index = pd.to_datetime(df_returns.index)

# Aligner les dates entre facteurs et rendements
common_dates = df_returns.index.intersection(df_factors_1996_2016.index)

# Filtrer les deux DataFrames sur les dates communes
returns_aligned = df_returns.loc[common_dates]
factors_aligned = df_factors_1996_2016.loc[common_dates]

# Extraire le taux sans risque (RF)
rf = factors_aligned['RF']

# Calculer les rendements excédentaires
excess_returns = returns_aligned.subtract(rf, axis=0)


In [ ]:
factors_aligned.head()

,Mkt-RF,SMB,HML,RMW,CMA,RF
Date,,,,,,
2007-03-23,0.0009,0.0013,0.0005,-0.0001,-0.0007,0.0002
2007-03-26,0.0006,-0.0007,-0.0017,-0.0002,-0.0006,0.0002
2007-03-27,-0.0062,-0.0016,-0.0005,-0.0012,-0.0018,0.0002
2007-03-28,-0.0074,0.0018,0.0002,-0.0010,0.0023,0.0002
2007-03-29,0.0029,-0.0013,0.0011,0.0041,0.0004,0.0002


In [ ]:

print(f"\nShape des rendements excédentaires : {excess_returns.shape}")
print(f"Période : {excess_returns.index.min()} à {excess_returns.index.max()}")
excess_returns.head()




Shape des rendements excédentaires : (2463, 473)
Période : 2007-03-23 00:00:00 à 2016-12-30 00:00:00


,TEL,FAST,SWK,IRM,LUV,CMI,NXPI,PODD,JCI,FRT,...,XYZ,CME,KDP,CCL,COR,WSM,MU,AVB,ABT,BK
Date,,,,,,,,,,,,,,,,,,,,,
2007-03-23,-0.001487,0.011808,0.005733,0.003564,0.001135,0.014529,-0.0002,-0.032155,0.003555,0.000677,...,-0.017032,-0.006247,-0.004122,0.005653,0.011248,-0.003939,-0.002782,0.001358,-0.011943,0.002010
2007-03-26,-0.001487,-0.019516,-0.010030,-0.004325,-0.008200,-0.009854,-0.0002,-0.032155,-0.005187,-0.020038,...,-0.017032,-0.005081,-0.004122,-0.001447,-0.001499,0.007017,0.001526,-0.020648,0.062555,0.003231
2007-03-27,-0.001487,-0.017364,-0.000200,-0.012624,-0.001544,-0.019496,-0.0002,-0.032155,-0.019937,-0.011494,...,-0.017032,-0.001668,-0.004122,-0.014349,-0.009862,-0.002493,0.034253,-0.023873,-0.011381,-0.008748
2007-03-28,-0.001487,0.000659,-0.004352,-0.006300,-0.011640,-0.016927,-0.0002,-0.032155,-0.004994,-0.004384,...,-0.017032,-0.015384,-0.004122,-0.020251,0.002803,0.002098,-0.000200,-0.020263,-0.015041,-0.008574
2007-03-29,-0.001487,-0.006207,0.002519,-0.000200,-0.004284,0.010188,-0.0002,-0.032155,0.003022,-0.002699,...,-0.017032,0.003314,-0.004122,0.009707,-0.000013,-0.007652,-0.014355,0.001152,0.000696,0.007251


In [ ]:
# Statistiques rapides des rendements excédentaires
print(f"Moyenne: {excess_returns.mean().mean():.6f}")
print(f"Std: {excess_returns.std().mean():.6f}")
print(f"Min: {excess_returns.min().min():.6f}")
print(f"Max: {excess_returns.max().max():.6f}")
print(f"Skewness moyen: {excess_returns.skew().mean():.4f}")
print(f"Kurtosis moyen: {excess_returns.kurtosis().mean():.4f}")
print(f"Actions: {excess_returns.shape[1]}, Jours: {excess_returns.shape[0]}")

Moyenne: 0.001165
Std: 0.022043
Min: -0.608008
Max: 1.023578
Skewness moyen: 0.3639
Kurtosis moyen: 15.3761
Actions: 473, Jours: 2463


## Construction des portefeuilles d’arbitrage


#### Fama-French

In [ ]:
def build_ff_arbitrage_vectorized(returns, factors, K=5, window=60):
    """
    Portefeuilles d'arbitrage Fama-French - Version vectorisee.

    Pour chaque jour t:
    - Train: fenetre glissante des 60 jours precedents
    - Loadings beta estimes par OLS: R = beta * F + epsilon
    - Residu: epsilon_t = R_t - beta * F_t
    - Phi = I - beta^T (R^T R)^{-1} R^T F

    Parametres:
    returns : DataFrame (T x N) - rendements excédentaires
    factors : DataFrame (T x K) - facteurs Fama-French
    K : int - 1, 3 ou 5 facteurs
    window : int - taille de la fenetre glissante (60)

    Retourne:
    residuals : DataFrame (T x N) - résidus epsilon
    phi : ndarray (T x N x N) - matrices de projection
    loadings : ndarray (T x N x K) - loadings beta
    r2 : ndarray (T x N) - coefficient de determination
    """

    factor_map = {1: ['Mkt-RF'], 3: ['Mkt-RF', 'SMB', 'HML'],
                  5: ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']}
    F = factors[factor_map[K]].values
    R = returns.values

    T, N = R.shape
    residuals = np.full((T, N), np.nan)
    r2 = np.full((T, N), np.nan)
    loadings = np.full((T, N, K), np.nan)
    phi = np.zeros((T, N, N))

    for t in range(window, T):
        # Fenetre d'entrainement
        R_train = R[t-window:t]  # (window, N)
        F_train = F[t-window:t]  # (window, K)
        F_test = F[t:t+1]        # (1, K)

        # Estimation des loadings beta = (F'F)^{-1} F'R
        FtF_inv = np.linalg.pinv(F_train.T @ F_train)
        beta = FtF_inv @ F_train.T @ R_train  # (K, N)

        # Residus
        pred = F_test @ beta
        residuals[t] = R[t] - pred[0]

        # R2
        pred_train = F_train @ beta
        ss_res = np.sum((R_train - pred_train)**2, axis=0)
        ss_tot = np.sum((R_train - np.mean(R_train, axis=0))**2, axis=0)
        r2[t] = 1 - ss_res / (ss_tot + 1e-8)

        loadings[t] = beta.T

        # Matrice Phi = I - beta^T (R^T R)^{-1} R^T F
        if N > K:
            # wF = (R'R)^{-1} R'F
            RtR_inv = np.linalg.pinv(R_train.T @ R_train)
            wF = RtR_inv @ R_train.T @ F_train  # (N, K)

            # Phi = I - beta^T wF^T
            phi[t] = np.eye(N) - beta.T @ wF.T

        if t % 500 == 0:
            print(f"Date {t}/{T}")

    residuals_df = pd.DataFrame(residuals, index=returns.index, columns=returns.columns)
    r2_df = pd.DataFrame(r2, index=returns.index, columns=returns.columns)

    return residuals_df, phi, loadings, r2_df

In [ ]:
residuals, phi, loadings, r2 = build_ff_arbitrage_vectorized(
    returns=excess_returns,
    factors=factors_aligned,
    K=5,
    window=60
)

Date 500/2463
Date 1000/2463
Date 1500/2463
Date 2000/2463


In [ ]:
# Filtrer les 60 premieres dates (NaN) et renommer
residuals_fama = residuals.iloc[60:].copy()
phi_fama = phi[60:]  # shape: (2403, 473, 473)
loadings_fama = loadings[60:]  # shape: (2403, 473, 5)
r2_fama = r2.iloc[60:].copy()

In [ ]:
residuals_fama.tail(5)

,TEL,FAST,SWK,IRM,LUV,CMI,NXPI,PODD,JCI,FRT,...,XYZ,CME,KDP,CCL,COR,WSM,MU,AVB,ABT,BK
Date,,,,,,,,,,,,,,,,,,,,,
2016-12-23,-0.001753,0.000851,0.005519,-0.000640,0.004770,0.000333,0.004277,-0.017473,0.000760,-0.004279,...,0.004884,-0.002838,0.003868,0.009765,0.009226,-0.024258,0.000714,0.007150,-0.005068,-0.000874
2016-12-27,-0.004823,-0.003461,-0.006214,-0.006737,-0.003926,-0.008379,0.008065,0.008342,-0.005707,0.000528,...,-0.011579,-0.006275,-0.002044,-0.011692,-0.000976,-0.003438,-0.001510,0.004899,0.005771,0.001146
2016-12-28,0.003705,-0.003285,-0.002612,0.002157,0.001925,0.007629,-0.004328,0.007183,-0.006090,-0.000717,...,0.006474,0.002396,0.003599,-0.001582,-0.003179,-0.005892,-0.007871,-0.004209,-0.000037,0.004357
2016-12-29,0.000881,-0.004739,-0.000941,-0.000797,0.003108,-0.003383,-0.003798,0.004124,-0.003624,0.010178,...,-0.022550,-0.000267,0.000281,-0.006522,0.016758,-0.004964,-0.021191,0.003284,0.005932,0.000728
2016-12-30,-0.004279,0.003136,0.001268,0.013191,-0.001687,0.012340,0.000487,0.010553,-0.007980,0.021638,...,0.000252,0.001595,0.003948,0.007649,-0.013220,0.009821,-0.005600,0.010769,0.007753,0.002162


#### ACP

In [ ]:
def build_pca_arbitrage_vectorized(returns, K=5, window=252, reg_window=60):
    """
    Portefeuilles d'arbitrage PCA - Version vectorisee.

    Pour chaque jour t:
    - Extraction des K facteurs PCA sur fenetre covariance (252 jours)
    - Regression OLS sur fenetre regression (60 jours)
    - Residu: epsilon_t = R_t - beta * F_t_pca
    - Phi = I - beta^T (R^T R)^{-1} R^T F_pca

    Parametres:
    returns : DataFrame (T x N) - rendements excédentaires
    K : int - 1, 3 ou 5 facteurs PCA
    window : int - fenetre pour covariance (252)
    reg_window : int - fenetre pour regression (60)

    Retourne:
    residuals : DataFrame (T x N) - résidus epsilon
    phi : ndarray (T x N x N) - matrices de projection
    loadings : ndarray (T x N x K) - loadings beta
    r2 : ndarray (T x N) - coefficient de determination
    """

    R = returns.values
    T, N = R.shape

    residuals = np.full((T, N), np.nan)
    r2 = np.full((T, N), np.nan)
    loadings = np.full((T, N, K), np.nan)
    phi = np.zeros((T, N, N))

    for t in range(reg_window, T):
        if t % 200 == 0:
            print(f"Date {t}/{T}")

        # 1. Extraction des facteurs PCA sur fenetre covariance
        cov_start = max(0, t - window)
        R_cov = R[cov_start:t]  # fenetre pour covariance

        # Normalisation
        mean = np.mean(R_cov, axis=0, keepdims=True)
        std = np.std(R_cov, axis=0, keepdims=True)
        std[std < 1e-8] = 1
        R_norm = (R_cov - mean) / std

        # ACP via SVD
        U, s, Vt = np.linalg.svd(R_norm, full_matrices=False)
        factors_pca = U[:, :K] @ np.diag(s[:K])  # facteurs non normalises

        # 2. Regression OLS sur fenetre reg_window
        reg_start = max(0, t - reg_window)
        R_reg = R[reg_start:t]
        F_reg = factors_pca[-reg_window:] if len(factors_pca) >= reg_window else factors_pca

        if F_reg.shape[0] < reg_window:
            continue

        # Loadings beta = (F'F)^{-1} F'R
        FtF_inv = np.linalg.pinv(F_reg.T @ F_reg)
        beta = FtF_inv @ F_reg.T @ R_reg  # (K, N)

        # Facteurs pour la prediction
        R_test_norm = (R[t:t+1] - mean) / std
        F_test = R_test_norm @ Vt[:K, :].T

        # Residu
        pred = F_test @ beta
        residuals[t] = R[t] - pred[0]

        # R2
        pred_train = F_reg @ beta
        ss_res = np.sum((R_reg - pred_train)**2, axis=0)
        ss_tot = np.sum((R_reg - np.mean(R_reg, axis=0))**2, axis=0)
        r2[t] = 1 - ss_res / (ss_tot + 1e-8)

        loadings[t] = beta.T

        # Matrice Phi = I - beta^T (R^T R)^{-1} R^T F
        if N > K and F_reg.shape[0] >= N:
            RtR_inv = np.linalg.pinv(R_reg.T @ R_reg)
            wF = RtR_inv @ R_reg.T @ F_reg
            phi[t] = np.eye(N) - beta.T @ wF.T

    residuals_df = pd.DataFrame(residuals, index=returns.index, columns=returns.columns)
    r2_df = pd.DataFrame(r2, index=returns.index, columns=returns.columns)

    return residuals_df, phi, loadings, r2_df



In [ ]:
# Execution avec K=5 facteurs PCA
residuals_pca, phi_pca, loadings_pca, r2_pca = build_pca_arbitrage_vectorized(
    returns=excess_returns,
    K=5,
    window=252,
    reg_window=60
)



Date 200/2463
Date 400/2463
Date 600/2463
Date 800/2463
Date 1000/2463
Date 1200/2463
Date 1400/2463
Date 1600/2463
Date 1800/2463
Date 2000/2463
Date 2200/2463
Date 2400/2463


In [ ]:
# Filtrer les 60 premieres dates
residuals_pca_filter = residuals_pca.iloc[60:].copy()
phi_pca_filter = phi_pca[60:]
loadings_pca_filter = loadings_pca[60:]
r2_pca_filter = r2_pca.iloc[60:].copy()


In [ ]:
residuals_pca_filter.head(2)

,TEL,FAST,SWK,IRM,LUV,CMI,NXPI,PODD,JCI,FRT,...,XYZ,CME,KDP,CCL,COR,WSM,MU,AVB,ABT,BK
Date,,,,,,,,,,,,,,,,,,,,,
2007-06-19,-0.012587,-0.004704,0.000426,-0.002531,0.013392,0.045511,-0.0002,-0.001141,-0.015190,-0.009552,...,-0.017032,-0.000312,-0.004122,0.000270,-0.005092,-0.017058,-0.009952,0.007909,0.002998,-0.000409
2007-06-20,-0.018604,0.020112,0.014030,0.006056,0.013961,0.039095,-0.0002,0.016571,-0.018362,-0.003007,...,-0.017041,0.006359,-0.004124,0.001698,-0.005322,0.001006,0.032610,-0.015623,0.005256,-0.004802


---
### Modelisation : signal + allocation

In [ ]:
import numpy as np
import pandas as pd

def ou_threshold_strategy_moments(residuals, phi, L=30, c_thresh=2.0, c_crit=0.5):
    """
    OU + Threshold avec estimation par méthode des moments (correcte).
    """

    T, N = residuals.shape
    w_epsilon = np.zeros((T, N))
    portfolio_returns = []

    # Résidu cumulé
    X = residuals.fillna(0).values.cumsum(axis=0)

    for t in range(L, T-1):  # important: T-1 pour éviter look-ahead
        X_window = X[t-L:t, :]   # info jusqu'à t-1

        X_curr = X_window[:-1, :]
        X_next = X_window[1:, :]

        # --- Estimation AR(1) (méthode des moments) ---
        denom = np.sum(X_curr**2, axis=0)
        denom[denom == 0] = 1e-8

        b = np.sum(X_curr * X_next, axis=0) / denom
        b = np.clip(b, 1e-6, 0.999)

        kappa = -np.log(b)

        a = np.mean(X_next - b * X_curr, axis=0)
        mu = a / (1 - b)

        # Résidus
        eps = X_next - (a + b * X_curr)

        var_eps = np.var(eps, axis=0)
        sigma = np.sqrt((2 * kappa * var_eps) / (1 - np.exp(-2 * kappa)))

        # --- Score OU ---
        X_L = X_window[-1, :]
        denom_score = sigma / np.sqrt(2 * kappa)
        denom_score[denom_score == 0] = 1e-8

        score = (X_L - mu) / denom_score

        # --- R² ---
        ss_res = np.sum(eps**2, axis=0)
        ss_tot = np.sum((X_next - np.mean(X_next, axis=0))**2, axis=0)
        r2 = 1 - ss_res / (ss_tot + 1e-8)

        # --- Allocation epsilon ---
        w_eps_t = np.zeros(N)
        mask = r2 > c_crit

        w_eps_t[mask] = np.where(
            score[mask] > c_thresh, -1,
            np.where(score[mask] < -c_thresh, 1, 0)
        )

        w_epsilon[t] = w_eps_t

        # --- Passage aux returns ---
        if t < len(phi):
            phi_t = phi[t]  # matrice (N x K)

            w_R = w_eps_t @ phi_t
            norm = np.sum(np.abs(w_R))

            if norm > 1e-8:
                w_R = w_R / norm

                # rendement à t+1 (pas de look-ahead)
                R_t1 = residuals.iloc[t+1].values
                port_ret = np.sum(w_R * R_t1)
                portfolio_returns.append(port_ret)

    port_returns = np.array(portfolio_returns)

    sharpe = (
        port_returns.mean() / port_returns.std() * np.sqrt(252)
        if port_returns.std() > 1e-8 else 0
    )

    return sharpe, port_returns

In [ ]:
print("Optimisation OU + Threshold (méthode des moments)")
print("="*50)

best_sharpe = -np.inf
best_params = None

for c_thresh in [1.0, 1.5, 2.0, 2.5, 3.0]:
    for c_crit in [0.3, 0.4, 0.5, 0.6, 0.7]:
        sharpe, _ = ou_threshold_strategy_moments(
            residuals_pca,
            phi_pca,
            L=30,
            c_thresh=c_thresh,
            c_crit=c_crit
        )

        print(f"c_thresh={c_thresh}, c_crit={c_crit} -> Sharpe={sharpe:.2f}")

        if sharpe > best_sharpe:
            best_sharpe = sharpe
            best_params = (c_thresh, c_crit)

print("="*50)
print(f"Meilleurs paramètres: c_thresh={best_params[0]}, c_crit={best_params[1]}")
print(f"Sharpe optimal: {best_sharpe:.2f}")

# stratégie finale
sharpe_final, port_returns_final = ou_threshold_strategy_moments(
    residuals_pca,
    phi_pca,
    L=30,
    c_thresh=best_params[0],
    c_crit=best_params[1]
)

print(f"\nSharpe final: {sharpe_final:.2f}")

Optimisation OU + Threshold (méthode des moments)
c_thresh=1.0, c_crit=0.3 -> Sharpe=0.00
c_thresh=1.0, c_crit=0.4 -> Sharpe=0.00
c_thresh=1.0, c_crit=0.5 -> Sharpe=0.00
c_thresh=1.0, c_crit=0.6 -> Sharpe=0.00
c_thresh=1.0, c_crit=0.7 -> Sharpe=0.00
c_thresh=1.5, c_crit=0.3 -> Sharpe=0.00
c_thresh=1.5, c_crit=0.4 -> Sharpe=0.00
c_thresh=1.5, c_crit=0.5 -> Sharpe=0.00
c_thresh=1.5, c_crit=0.6 -> Sharpe=0.00
c_thresh=1.5, c_crit=0.7 -> Sharpe=0.00
c_thresh=2.0, c_crit=0.3 -> Sharpe=0.00
c_thresh=2.0, c_crit=0.4 -> Sharpe=0.00
c_thresh=2.0, c_crit=0.5 -> Sharpe=0.00
c_thresh=2.0, c_crit=0.6 -> Sharpe=0.00
c_thresh=2.0, c_crit=0.7 -> Sharpe=0.00
c_thresh=2.5, c_crit=0.3 -> Sharpe=0.00
c_thresh=2.5, c_crit=0.4 -> Sharpe=0.00
c_thresh=2.5, c_crit=0.5 -> Sharpe=0.00
c_thresh=2.5, c_crit=0.6 -> Sharpe=0.00
c_thresh=2.5, c_crit=0.7 -> Sharpe=0.00
c_thresh=3.0, c_crit=0.3 -> Sharpe=0.00
c_thresh=3.0, c_crit=0.4 -> Sharpe=0.00
c_thresh=3.0, c_crit=0.5 -> Sharpe=0.00
c_thresh=3.0, c_crit=0.6 -> Sh

In [ ]:
print("Nb trades:", np.sum(np.abs(w_epsilon)))
print("Nb returns:", len(portfolio_returns))
print("Mean return:", np.mean(portfolio_returns) if len(portfolio_returns)>0 else None)
print("Std return:", np.std(portfolio_returns) if len(portfolio_returns)>0 else None)

Nb trades: nan
Nb returns: 0
Mean return: None
Std return: None
